# Topic 3 — Logistic Regression

> **Goal of this notebook:** understand how logistic regression turns a linear model into a **classifier** using the sigmoid function and cross-entropy loss, then apply it to predict whether a tumour is malignant or benign.

**Contents**
1. Concept & intuition
2. The mathematics (sigmoid, log-loss, gradient)
3. Assumptions
4. The dataset: Breast Cancer Wisconsin
5. Exploratory data analysis
6. Training with scikit-learn
7. Evaluation (confusion matrix, precision/recall, ROC-AUC)
8. From scratch (sigmoid + gradient descent)
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

Logistic regression is a **classification** algorithm (despite the name). Instead of predicting a number, it predicts the **probability** that an example belongs to a class.

It starts with the same linear combination as linear regression, $z = \mathbf{w}\cdot\mathbf{x} + b$, but then squashes it through the **sigmoid** function so the output lands between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

If $\sigma(z) \geq 0.5$ we predict class 1, otherwise class 0. The boundary where $z = 0$ is the **decision boundary** — a line/hyperplane separating the classes.

## 2. The Mathematics

**Prediction:** $\hat{p} = \sigma(\mathbf{w}\cdot\mathbf{x} + b)$

**Loss — Binary Cross-Entropy (log-loss):** we can't use MSE (it's non-convex with the sigmoid). Instead:

$$J = -\frac{1}{m}\sum_{i=1}^{m}\Big[ y^{(i)}\log \hat{p}^{(i)} + (1-y^{(i)})\log(1-\hat{p}^{(i)}) \Big]$$

This heavily penalises confident wrong predictions. Conveniently, its gradient has the same clean form as linear regression:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_i (\hat{p}^{(i)} - y^{(i)})\,x_j^{(i)}$$

so we again train with gradient descent.

## 3. Assumptions

1. The **log-odds** (logit) are a linear function of the features.
2. Observations are independent.
3. Little multicollinearity among features.
4. Reasonably large sample size for stable estimates.

Features don't need to be normally distributed, but **scaling helps** optimisation converge.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, accuracy_score)

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Breast Cancer Wisconsin

A classic binary-classification dataset built into scikit-learn: 569 tumours described by 30 features computed from cell-nucleus images. Target: **malignant (0)** vs **benign (1)**.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print('Shape:', X.shape)
print('Classes:', dict(zip(data.target_names, np.bincount(y))))
X.iloc[:, :5].head()

## 5. Exploratory Data Analysis

Which features separate the two classes best? Compare mean values per class.

In [ ]:
df = X.copy()
df['target'] = y
means = df.groupby('target').mean().T
means['abs_diff'] = (means[0] - means[1]).abs()
print('Top 5 most discriminative features (by class-mean gap):')
print(means.sort_values('abs_diff', ascending=False).head())

In [ ]:
feat = 'worst concave points'
plt.hist(X[feat][y==0], bins=30, alpha=0.6, label='malignant')
plt.hist(X[feat][y==1], bins=30, alpha=0.6, label='benign')
plt.xlabel(feat); plt.ylabel('count')
plt.title(f'Class separation on "{feat}"')
plt.legend(); plt.show()

## 6. Training with scikit-learn

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=5000)
clf.fit(X_train_s, y_train)
print('Training accuracy:', round(clf.score(X_train_s, y_train), 4))
print('Test accuracy:    ', round(clf.score(X_test_s, y_test), 4))

## 7. Evaluation

Accuracy alone is misleading for classification — especially with imbalance or asymmetric costs (missing a malignant tumour is far worse than a false alarm). We use a **confusion matrix**, **precision/recall/F1**, and the **ROC curve / AUC**.

In [ ]:
y_pred = clf.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:')
print(pd.DataFrame(cm, index=['true malignant', 'true benign'],
                   columns=['pred malignant', 'pred benign']))
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# ROC curve: probability of the positive class (benign = 1)
y_proba = clf.predict_proba(X_test_s)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.plot(fpr, tpr, lw=2, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend(); plt.show()

## 8. Logistic Regression From Scratch

Implement the sigmoid and gradient descent to confirm we understand the mechanics.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class LogisticRegressionGD:
    def __init__(self, lr=0.1, n_iters=2000):
        self.lr, self.n_iters = lr, n_iters

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n); self.b = 0.0
        for _ in range(self.n_iters):
            p = sigmoid(X @ self.w + self.b)
            error = p - y
            self.w -= self.lr * (X.T @ error) / m
            self.b -= self.lr * error.mean()
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
scratch = LogisticRegressionGD(lr=0.1, n_iters=2000).fit(X_train_s, np.asarray(y_train))
print('From-scratch test accuracy:', round(accuracy_score(y_test, scratch.predict(X_test_s)), 4))
print('scikit-learn test accuracy:', round(accuracy_score(y_test, y_pred), 4))

## 9. Pros, Cons & When to Use

**Pros**
- Outputs calibrated **probabilities**, not just labels.
- Fast, simple, and highly interpretable (coefficients = log-odds impact).
- Strong baseline for any classification problem.

**Cons**
- Assumes a linear decision boundary — underfits complex patterns.
- Sensitive to outliers and irrelevant features.
- Needs feature scaling for fast convergence.

**When to use**
- Binary (or, via softmax, multiclass) classification with roughly linear separation.
- When you need probability estimates and interpretability.
- As the baseline before trying trees, SVMs, or boosting.

## 10. Summary

- Logistic regression applies the **sigmoid** to a linear model to output class probabilities.
- It is trained by minimising **binary cross-entropy** via gradient descent.
- On the Breast Cancer data it reached ~97% accuracy with a high ROC-AUC, and the confusion matrix lets us reason about the costly errors (missed malignancies).
- Our from-scratch implementation matched scikit-learn, confirming the mechanics.